In [1]:
import json
from pathlib import Path
import sys
import os


project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

print("Project root added:", project_root)





configs_dir = Path("ensemble_configs")

top_configs = []
for i in range(1, 4):
    cfg_path = configs_dir / f"top_{i}.json"
    with open(cfg_path, "r") as f:
        top_configs.append(json.load(f))

top_configs


import pandas as pd
import torch


train_df = pd.read_csv("../data/processed/train.csv")
test_df  = pd.read_csv("../data/processed/test.csv")


target_col = "class"


feature_cols = [c for c in train_df.columns if c != target_col]


device = "cuda" if torch.cuda.is_available() else "cpu"

train_df.head(), test_df.head(), device


Project root added: /home/dw/Desktop/mlp-openml-project


(      x-box     y-box     width      high     onpix     x-bar     y-bar  \
 0 -0.077515  0.240085  0.377598  0.228897  1.554544  0.044761  0.198450   
 1 -0.077515  0.866333 -0.657465  0.687577  0.625818  0.044761 -0.232085   
 2 -0.608236 -1.325534 -1.174997 -1.605826 -1.231632 -0.941455  0.198450   
 3 -1.138957 -1.325534 -1.692528 -1.605826 -1.231632 -0.448347  0.198450   
 4 -1.138957  0.240085 -0.657465 -0.229784 -1.231632  0.537869 -0.662620   
 
       x2bar     y2bar     xybar     x2ybr     xy2br     x-ege     xegvy  \
 0  0.539982 -0.519954 -0.545682 -0.552138  0.041398  1.229098  0.409524   
 1  2.830933 -1.815023 -0.950605 -0.171197  0.041398  0.799842 -0.226042   
 2  0.158156  0.775114  0.264164  0.590685  0.041398 -0.058671  0.409524   
 3  0.158156 -0.519954  0.264164  0.590685  0.518171 -0.487928 -0.226042   
 4 -0.605494 -1.815023 -0.545682 -2.456842  0.041398 -0.487928 -0.861608   
 
       y-ege     yegvx  class  
 0  1.262215 -0.471743     12  
 1 -1.495610  0.1285

In [2]:
from torch.utils.tensorboard import SummaryWriter
from sklearn.model_selection import StratifiedKFold
import torch
import os

def train_cv_with_logging(config_id, config, train_df, feature_cols, target_col, device, n_splits=5, n_epochs=20):
    """
    config_id: id for config, one of the best (1,2,3)
    config: dictionary with hiperparameters (hidden_layers, dropout, lr, batch_size, grad_clip)
    """
    log_dir = f"runs/config_{config_id}"
    os.makedirs(log_dir, exist_ok=True)

    writer = SummaryWriter(log_dir=log_dir)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    fold_models = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df[target_col])):
        print(f"Config {config_id} – Fold {fold}")

        train_fold = train_df.iloc[train_idx]
        val_fold   = train_df.iloc[val_idx]

        train_loader = create_dataloader(train_fold, feature_cols, target_col,
                                         batch_size=config["batch_size"], shuffle=True)
        val_loader   = create_dataloader(val_fold,   feature_cols, target_col,
                                         batch_size=config["batch_size"], shuffle=False)

        model = build_mlp(
            config={"hidden_layers": config["hidden_layers"], "dropout": config["dropout"]},
            input_dim=len(feature_cols),
            output_dim=train_df[target_col].nunique()
        ).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])
        criterion = torch.nn.CrossEntropyLoss()

        best_val_loss = float("inf")
        best_state = None

        for epoch in range(n_epochs):
            model.train()
            train_loss = train_one_epoch(model, train_loader, optimizer, device, config["grad_clip"])
            val_loss, _ = validate(model, val_loader, device)

            # logs for tensorboard
            global_step = fold * n_epochs + epoch
            writer.add_scalar(f"fold_{fold}/train_loss", train_loss, global_step)
            writer.add_scalar(f"fold_{fold}/val_loss",   val_loss,   global_step)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = model.state_dict()

        # saving best modelfor each fold
        fold_model_path = f"models/config_{config_id}_fold_{fold}.pt"
        os.makedirs("models", exist_ok=True)
        torch.save(best_state, fold_model_path)
        fold_models.append(fold_model_path)

    writer.close()
    return fold_models


In [3]:


# data loading, dataset, dataloader
from src.data import (
    create_dataloader,
    TabularDataset,
    clean_data,
    fit_scaler,
    apply_scaler,
)

# model
from src.models import build_mlp

# training utilities
from src.train import (
    train_one_epoch,
    validate,
)

all_models = {}

for config_id, config in enumerate(top_configs, start=1):
    print(f"\n=== Training for config {config_id} ===")
    fold_models = train_cv_with_logging(
        config_id=config_id,
        config=config,
        train_df=train_df,
        feature_cols=feature_cols,
        target_col=target_col,
        device=device,
        n_splits=5,
        n_epochs=20 
    )
    all_models[config_id] = fold_models

all_models

{
    1: ["models/config_1_fold_0.pt", ..., "models/config_1_fold_4.pt"],
    2: [...],
    3: [...]
}



=== Training for config 1 ===
Config 1 – Fold 0
Config 1 – Fold 1
Config 1 – Fold 2
Config 1 – Fold 3
Config 1 – Fold 4

=== Training for config 2 ===
Config 2 – Fold 0
Config 2 – Fold 1
Config 2 – Fold 2
Config 2 – Fold 3
Config 2 – Fold 4

=== Training for config 3 ===
Config 3 – Fold 0
Config 3 – Fold 1
Config 3 – Fold 2
Config 3 – Fold 3
Config 3 – Fold 4


{1: ['models/config_1_fold_0.pt', Ellipsis, 'models/config_1_fold_4.pt'],
 2: [Ellipsis],
 3: [Ellipsis]}